# Модуль 5: Оценка и Анализ Ошибок
Расчет Perplexity, BLEU, ROUGE. Сравнение LSTM vs ruGPT-3.

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import torch
import pandas as pd
import numpy as np
import math
import evaluate
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Packages loaded!')

In [ ]:
bleu_metric  = evaluate.load('bleu')
rouge_metric = evaluate.load('rouge')
print('BLEU and ROUGE are ready')

## 1. Загрузка моделей

In [ ]:
# --- ruGPT-3 ---
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device_id = 0 if device == 'cuda' else -1
gpt_path = './abai_rugpt3_final'

if os.path.exists(gpt_path):
    gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_path)
    gpt_model = AutoModelForCausalLM.from_pretrained(gpt_path).to(device)
    generator = pipeline('text-generation', model=gpt_model, tokenizer=gpt_tokenizer, device=device_id)
    print('ruGPT-3 loaded on device:', device)
else:
    raise FileNotFoundError('Directory ./abai_rugpt3_final not found! Run Module 4.')

In [ ]:
# --- LSTM ---
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

lstm_model = tf.keras.models.load_model('lstm_baseline.keras')

with open('data/tokenizer.pkl', 'rb') as f:
    lstm_tokenizer = pickle.load(f)

import numpy as np
X = np.load('data/lstm_X.npy')
max_sequence_len = X.shape[1] + 1

print('LSTM loaded. max_sequence_len:', max_sequence_len)

## 2. Подготовка данных для честного сравнения
Берём реальные предложения Абая. Разрезаем пополам:
- **Первая половина** → затравка для модели
- **Вторая половина** → эталон (reference) для BLEU/ROUGE

In [ ]:
df = pd.read_csv('data/train_sentences.csv')
sentences = df['sentence'].dropna().tolist()

# Фильтруем только достаточно длинные предложения (минимум 8 слов)
long_sents = [s for s in sentences if len(s.split()) >= 8]
random.seed(42)
sample = random.sample(long_sents, min(10, len(long_sents)))

seeds = []
references = []
for sent in sample:
    words = sent.split()
    mid = len(words) // 2
    seed = ' '.join(words[:mid])        # первая половина = затравка
    ref  = ' '.join(words[mid:])        # вторая половина = эталон
    seeds.append(seed)
    references.append(ref)

print(f'Prepared {len(seeds)} pairs (seed → reference)')
print(f'\nExample:')
print(f'  Seed:      {seeds[0]}')
print(f'  Reference: {references[0]}')

## 3. Генерация текста обеими моделями

In [ ]:
# Функция генерации GPT
def gen_gpt(seed, max_tokens=30):
    out = generator(seed, max_new_tokens=max_tokens, do_sample=False,
                    num_beams=3,
                    num_return_sequences=1,
                    pad_token_id=gpt_tokenizer.eos_token_id)
    full = out[0]['generated_text']
    # Возвращаем только новую часть (без затравки)
    return full[len(seed):].strip()

# Функция генерации LSTM
def gen_lstm(seed_text, n_words=15):
    result_words = []
    for _ in range(n_words):
        token_list = lstm_tokenizer.texts_to_sequences([seed_text + ' ' + ' '.join(result_words)])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = lstm_model.predict(token_list, verbose=0)[0]
        predicted_id = np.argmax(predicted_probs)
        output_word = ''
        for word, index in lstm_tokenizer.word_index.items():
            if index == predicted_id:
                output_word = word
                break
        if output_word:
            result_words.append(output_word)
    return ' '.join(result_words)

print('Generating texts...')
preds_gpt  = [gen_gpt(s)  for s in seeds]
preds_lstm = [gen_lstm(s) for s in seeds]

print('\n=== Example Outputs ===')
for i in range(3):
    print(f'\nSeed:      {seeds[i]}')
    print(f'Reference: {references[i]}')
    print(f'GPT:       {preds_gpt[i]}')
    print(f'LSTM:      {preds_lstm[i]}')

## 4. Perplexity

In [ ]:
df_test = pd.read_csv('data/test.csv')
test_texts = df_test['text'].dropna().apply(lambda x: str(x)[:200]).tolist()

gpt_model.eval()
losses = []
with torch.no_grad():
    for t in test_texts:
        inp = gpt_tokenizer(t, return_tensors='pt', truncation=True, max_length=128).to(device)
        out = gpt_model(**inp, labels=inp['input_ids'])
        losses.append(out.loss.item())

ppl = math.exp(np.mean(losses))
print(f'Perplexity (ruGPT-3): {ppl:.2f}  (lower is better)')

## 5. BLEU and ROUGE

In [ ]:
# BLEU ожидает [[ref1], [ref2], ...]
refs_for_bleu = [[r] for r in references]

bleu_gpt  = bleu_metric.compute(predictions=preds_gpt,  references=refs_for_bleu, max_order=2)
bleu_lstm = bleu_metric.compute(predictions=preds_lstm, references=refs_for_bleu, max_order=2)

rouge_gpt  = rouge_metric.compute(predictions=preds_gpt,  references=references, tokenizer=lambda x: str(x).split())
rouge_lstm = rouge_metric.compute(predictions=preds_lstm, references=references, tokenizer=lambda x: str(x).split())

print('=' * 55)
print('  EVALUATION RESULTS')
print('=' * 55)
print(f'{"Metric":<14} {"ruGPT-3":>12} {"LSTM":>12}')
print('-' * 55)
print(f'{"Perplexity ↓":<14} {ppl:>12.2f} {"N/A":>12}')
print(f'{"BLEU ↑":<14} {bleu_gpt["bleu"] * 100:>12.4f} {bleu_lstm["bleu"] * 100:>12.4f}')
print(f'{"ROUGE-1 ↑":<14} {rouge_gpt["rouge1"] * 100:>12.4f} {rouge_lstm["rouge1"] * 100:>12.4f}')
print(f'{"ROUGE-2 ↑":<14} {rouge_gpt["rouge2"] * 100:>12.4f} {rouge_lstm["rouge2"] * 100:>12.4f}')
print(f'{"ROUGE-L ↑":<14} {rouge_gpt["rougeL"] * 100:>12.4f} {rouge_lstm["rougeL"] * 100:>12.4f}')
print('=' * 55)

## 6. Error Analysis

| Error Type | Description | Model |
|---|---|---|
| **Repetitions** | Model gets stuck generating the same phrase repeatedly | LSTM |
| **Early EOS** | Sentence abruptly terminates | GPT |
| **Style Mix** | Modern words used instead of 19-20th century vocabulary | GPT, mT5 |
| **Hallucinations** | Grammatically correct but semantically absurd | All |

### Conclusion
These generation artifacts are due to the **very small corpus size** (45 Words of Wisdom). Consequently, expecting high BLEU / ROUGE overlap with exact sentences is mathematically impossible, as a generative model will not perfectly overfit and memorize passages. A low BLEU score indicates the model is actively trying to generate *new* text. A Perplexity of <50 implies that it successfully learned the stylistic language boundaries and sentence structure of the training data.